<a href="https://colab.research.google.com/github/vahagngrigoryan2006/flyrank-internship-ml/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vahagngrigoryan2006/flyrank-internship-ml/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
from google.colab import userdata
from datasets import load_dataset
import pandas as pd

# Retrieve token securely from Colab Secrets
HF_TOKEN = userdata.get('HF_TOKEN')

import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"
DIM_CONTENT   = f"read_parquet(\'{WAREHOUSE}/dim_content.parquet\')"
FACT_MAY_JUNE  = (
    f"read_parquet(['{WAREHOUSE}/fact_content_daily_performance/month=2026-04/*.parquet', "
    f"'{WAREHOUSE}/fact_content_daily_performance/month=2026-05/*.parquet', "
    f"'{WAREHOUSE}/fact_content_daily_performance/month=2026-06/*.parquet'])"
)

features = con.sql(f"""
    WITH bounds AS (
        SELECT DATE \'2026-07-01\' AS as_of_date
    ),
    per_item AS (
        SELECT f.client_hash_id, f.content_hash_id,
               MIN(f.report_date) AS first_seen,
               SUM(CASE WHEN f.report_date >= b.as_of_date - INTERVAL 30 DAY
                        THEN f.gsc_impressions ELSE 0 END) AS last_30_impressions,
               SUM(CASE WHEN f.report_date <  b.as_of_date - INTERVAL 30 DAY AND f.report_date >= b.as_of_date - INTERVAL 60 DAY
                        THEN f.gsc_impressions ELSE 0 END) AS prev_30_impressions,
               AVG(CASE WHEN f.report_date <  b.as_of_date - INTERVAL 30 DAY AND f.report_date >= b.as_of_date - INTERVAL 60 DAY
                        THEN f.gsc_avg_position END)       AS prev_30_avg_position
        FROM {FACT_MAY_JUNE} f, bounds b
        GROUP BY 1, 2
    )
    FROM per_item p
    JOIN {DIM_CONTENT} d USING (content_hash_id)
    WHERE p.first_seen <= DATE \'2026-07-01\' - INTERVAL 60 DAY   -- guard (a): full prev_30 history
      AND p.prev_30_impressions >= 100                             -- guard (b): activity floor
""").df()

decision_moment = pd.Timestamp("2026-06-01")
features["content_created_date"] = pd.to_datetime(features["content_created_date"])
features["content_updated_date"] = pd.to_datetime(features["content_updated_date"])
features["content_age_days_at_decision"] = (decision_moment - features["content_created_date"]).dt.days
features["days_since_last_update_at_decision"] = (decision_moment - features["content_updated_date"]).dt.days

features = features[features["days_since_last_update_at_decision"] > 0] # the guard (c)

print(f"Feature frame: {len(features):,} content items surviving both guards.")

print(features.isna().sum())



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame: 55,904 content items surviving both guards.
client_hash_id                            0
content_hash_id                           0
first_seen                                0
last_30_impressions                       0
prev_30_impressions                       0
prev_30_avg_position                      0
client_hash_id_1                          0
keyword_hash_id                        1032
url_hash_id                               0
keyword_char_count                        0
keyword_token_count                       0
url_char_count                            0
content_created_date                      0
content_updated_date                      0
content_type                              0
search_volume                          1183
competition                            1183
competition_level                      1593
cpc                                    1183
main_intent                            1444
backlinks                             17441
category_count   

## 2. Feature notes (meaning, missing, categorical, available-when?)

All the features listed below exist before the decision day.

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

**prev_30_impressions**: The number of impressions during previous 60th day to 30th day period. No missing values

**prev_30_avg_position**: The average position of the content during previous 60th day to 30th day period. No missing values.

**content_age_days_at_decision**: days between the created day of the content and the decision day (30th day)

**days_since_last_update_at_decision**: days between the last update and the decision day (30th day) (`>0`)

**keyword_char_count**: char count of the keyword. No missing values.

**keyword_token_count**: number of tokens in the keyword.

**content_type**: Classification of the content (keyword article, feedly article, comparison article). No missing values.

**search_volume**: 	Search-volume estimate for the page's target keyword. The value is missing when there is no keyword data available. All missing values were replaced with zero, while a new boolean column **no_keyword_data** indicates the missing information.

**competition**: Keyword competition score, `0–1`. The value is missing when there is no keyword data available. All missing values were replaced with zero, while a new boolean column **no_keyword_data** indicates the missing information.

**cpc**: Cost-per-click estimate for the target keyword. The value is missing when there is no keyword data available. All missing values were replaced with zero, while a new boolean column **no_keyword_data** indicates the missing information.

**main_intent**: Primary search intent of the keyword (informational, transactional, commercial, navigational). Missing values are considered a separate category.

**backlinks**: Count of external inbound links pointing to the content’s URL. All missing values were replaced with zero, while a new boolean column **backlinks_na** indicates the missing information.

**category_count**: Number of topical categories assigned to the content. No missing values.

**word_count_tier** Total word count of the content body, binned into the categories `<1000, 1000-2000, 2000-3500, 3500+, NA`.

**char_count_tier**: Total character count of the content body, binned into the categories `<8000, 8000-15000, 15000-25000, 25000+, NA`.

**no_keyword_data**: boolean, indicates whether keyword data is missing.

**backlinks_na**: boolean, indicates whether backlink information is missing.

In [34]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

selected_features = [
    "prev_30_impressions",
    "prev_30_avg_position",
    "content_age_days_at_decision",
    "days_since_last_update_at_decision",
    "keyword_char_count",
    "keyword_token_count",
    "content_type",
    "search_volume",
    "competition",
    "cpc",
    "main_intent",
    "backlinks",
    "category_count",
    "char_count",
    "word_count"
]


df = features[selected_features]

import numpy as np

bins = [-np.inf, 1000, 2000, 3500, np.inf]
labels = ["<1000", "1000-2000", "2000-3500", "3500+"]

df["word_count_tier"] = pd.cut(
    df["word_count"],
    bins=bins,
    labels=labels,
    right=False
)

df["word_count_tier"] = (
    df["word_count_tier"]
    .astype("object")
    .fillna("NA")
)

bins = [-np.inf, 8000, 15000, 25000, np.inf]
labels = ["<8000", "8000-15000", "15000-25000", "25000+"]

df["char_count_tier"] = pd.cut(
    df["char_count"],
    bins=bins,
    labels=labels,
    right=False
)

df["char_count_tier"] = (
    df["char_count_tier"]
    .astype("object")
    .fillna("NA")
)

df = df.drop(columns = ["word_count", "char_count"])

# The three columns are NULL whenever there is no keyword data.
# We replace those NAs with zero, while also creating a new column which indicates the absence of keyword data
df["no_keyword_data"] = df["competition"].isna().astype(int)
df["search_volume"] = df["search_volume"].fillna(0)
df["competition"] = df["competition"].fillna(0)
df["cpc"] = df["cpc"].fillna(0)

# Main intent is NULL whenever it is unknown. NAs in a categorical column can be given a distinct category
df["main_intent"] = df["main_intent"].fillna("NA")

# We create a column to indicate the absence of information about backlinks, and
# replace NAs in backlinks with zeros
df["backlinks_na"] = df["backlinks"].isna().astype(int)
df["backlinks"] = df["backlinks"].fillna(0)



print("The missing values from modifies features:")
print(df.isna().sum())



The missing values from modifies features:
prev_30_impressions                   0
prev_30_avg_position                  0
content_age_days_at_decision          0
days_since_last_update_at_decision    0
keyword_char_count                    0
keyword_token_count                   0
content_type                          0
search_volume                         0
competition                           0
cpc                                   0
main_intent                           0
backlinks                             0
category_count                        0
word_count_tier                       0
char_count_tier                       0
no_keyword_data                       0
backlinks_na                          0
dtype: int64


/tmp/ipykernel_1756/3929916601.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["word_count_tier"] = pd.cut(
/tmp/ipykernel_1756/3929916601.py:37: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["word_count_tier"] = (
/tmp/ipykernel_1756/3929916601.py:46: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/index

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.